In [5]:
"""
Extracts table-ready values from execution_rate_summary.json files.

Expected file naming convention:
    {exp_key}_{model_key}_execution_rate_summary.json
    e.g. exp1_gpt4o_execution_rate_summary.json

Output:
    table_values.csv  — flat rows, one per experiment/model combination
"""

import json
import csv
from pathlib import Path
from collections import defaultdict

# =============================================================================
# CONFIGURATION
# =============================================================================

OUTPUT_CSV = "/Volumes/Rachna-HD/AssertAnalysisResults/RQ3_table_values.csv"

# Map (experiment_label, model_label) → path to execution_rate_summary.json
# Labels appear as-is in the CSV output — edit freely.
INPUT_FILES: dict[tuple[str, str], str] = {
    ("Exp3 (Minimal)", "GPT-4o"):  "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT4o/Assert_log_analysis/execution_rate_summary.json",
    ("Exp3 (Minimal)", "Qwen3-coder-480b"):   "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/Qwen_480b_cloud/Assert_log_analysis/execution_rate_summary.json",
    ("Exp3 (Minimal)", "GPT-OSS-120b"): "/Volumes/Rachna-HD/AssertAnalysisResults/Exp3LLMOutput/GPT_OSS_120b/Assert_log_analysis/execution_rate_summary.json",
    ("Exp6 (Method)",  "GPT-4o"):  "/Volumes/Rachna-HD/AssertAnalysisResults/Exp6LLMOutput/GPT4o/Assert_log_analysis/execution_rate_summary.json",
    ("Exp6 (Method)",  "Qwen3-coder-480b"):   "/Volumes/Rachna-HD/AssertAnalysisResults/Exp6LLMOutput/Qwen_480b_cloud/Assert_log_analysis/execution_rate_summary.json",
    ("Exp6 (Method)",  "GPT-OSS-120b"): "/Volumes/Rachna-HD/AssertAnalysisResults/Exp6LLMOutput/GPT_OSS_120b/Assert_log_analysis/execution_rate_summary.json",
    ("Exp7 (Class)",   "GPT-4o"):  "/Volumes/Rachna-HD/AssertAnalysisResults/Exp7LLMOutput/GPT4o/Assert_log_analysis/execution_rate_summary.json",
    ("Exp7 (Class)",   "Qwen3-coder-480b"):   "/Volumes/Rachna-HD/AssertAnalysisResults/Exp7LLMOutput/Qwen_480b_cloud/Assert_log_analysis/execution_rate_summary.json",
    ("Exp7 (Class)",   "GPT-OSS-120b"): "/Volumes/Rachna-HD/AssertAnalysisResults/Exp7LLMOutput/GPT_OSS_120b/Assert_log_analysis/execution_rate_summary.json",
}

# =============================================================================
# EXCEPTION CATEGORIES
# =============================================================================

# Maps JSON key → display/column name.
# The JSON is expected to use the exact exception class names as keys.
_EXCEPTION_CATEGORIES = {
    "NullPointerException":         "NullPointerException",
    "ClassNotFoundException":       "ClassNotFoundException",
    "NoClassDefFoundError":         "NoClassDefFoundError",
    "NoSuchMethodError":            "NoSuchMethodError",
    "NoSuchMethodException":        "NoSuchMethodException",
    "NoSuchFieldError":             "NoSuchFieldError",
    "AbstractMethodError":          "AbstractMethodError",
    "IncompatibleClassChangeError": "IncompatibleClassChangeError",
    "ClassCastException":           "ClassCastException",
    "IllegalAccessError":           "IllegalAccessError",
    "IllegalArgumentException":     "illegal_argument",
    "VerifyError":                  "VerifyError",
    "LinkageError":                 "LinkageError",
    "ExceptionInInitializerError":  "ExceptionInInitializerError",
    "AssertionError":               "AssertionError",
    "AssertionFailedError":         "AssertionFailedError",
    "InvocationTargetException":    "InvocationTargetException",
    # "other" is always appended separately
}


def _extract_exc(ec: dict, prefix: str) -> dict:
    """
    Extract exception category counts from a breakdown dict.
    Uses _EXCEPTION_CATEGORIES for known keys, then sums everything
    else into '{prefix}_other'.
    """
    result = {}
    known_json_keys = set(_EXCEPTION_CATEGORIES.keys())

    for json_key, col_name in _EXCEPTION_CATEGORIES.items():
        result[f"{prefix}_{col_name}"] = ec.get(json_key, 0)

    # Collect all unrecognised keys + any explicit "other" entry
    other = ec.get("other", 0)
    for k, v in ec.items():
        if k not in known_json_keys and k != "other":
            other += v

    result[f"{prefix}_other"] = other
    return result


# =============================================================================
# EXTRACTOR
# =============================================================================

def extract(summary: dict) -> dict:
    """
    Pull all table-relevant fields from one execution_rate_summary.json.
    Returns a flat dict of display-ready values.
    """
    a1 = summary.get("analysis_1_has_real_calls_true", {})
    a2 = summary.get("analysis_2_has_real_calls_false", {})

    # ── Analysis 1 top-level ─────────────────────────────────────────────
    total_has     = a1.get("total_testfiles", 0)
    exec_asserts  = a1.get("testfiles_where_assert_executed", 0)
    failed_before = a1.get("testfiles_failed_before_assert", 0)

    # ── Assert executed try/catch breakdown ──────────────────────────────
    ae_tc = a1.get("assert_executed_trycatch_breakdown", {})
    assert_exec_in_tc  = ae_tc.get("files_with_assert_in_trycatch", 0)
    assert_exec_out_tc = ae_tc.get("files_with_assert_outside_trycatch", 0)

    # ── Failure lifecycle stage (Analysis 1) ────────────────────────────
    fk1 = a1.get("failure_kind_breakdown", {})
    a1_static_init = fk1.get("static_init", 0)
    a1_test_method = fk1.get("test_method", 0)
    a1_constructor = fk1.get("constructor", 0)
    a1_unknown     = fk1.get("unknown", 0)
    a1_other_stage = fk1.get("other", 0)

    # ── Exception category (Analysis 1) ──────────────────────────────────
    a1_exc = _extract_exc(a1.get("exception_category_breakdown", {}), "a1")

    # ── Analysis 2 top-level ─────────────────────────────────────────────
    total_no = a2.get("total_testfiles", 0)

    # ── Failure lifecycle stage (Analysis 2) ────────────────────────────
    fk2 = a2.get("failure_kind_breakdown", {})
    a2_static_init = fk2.get("static_init", 0)
    a2_test_method = fk2.get("test_method", 0)
    a2_constructor = fk2.get("constructor", 0)
    a2_unknown     = fk2.get("unknown", 0)
    a2_other_stage = fk2.get("other", 0)

    # ── Exception category (Analysis 2) ──────────────────────────────────
    a2_exc = _extract_exc(a2.get("exception_category_breakdown", {}), "a2")

    a2_other_files = a2.get("other_files_for_investigation", [])

    result = {
        # ── totals ────────────────────────────────────────────────────
        "total_files":            total_has + total_no,
        "a1_total_files":         total_has,
        "a2_total_files":         total_no,

        # ── assert execution ──────────────────────────────────────────
        "assert_executed":        exec_asserts,
        "assert_executed_in_tc":  assert_exec_in_tc,
        "assert_executed_out_tc": assert_exec_out_tc,
        "failed_before_assert":   failed_before,

        # ── A1 lifecycle stage ────────────────────────────────────────
        "a1_static_init":         a1_static_init,
        "a1_constructor":         a1_constructor,
        "a1_test_method":         a1_test_method,
        "a1_unknown":             a1_unknown,
        "a1_other_stage":         a1_other_stage,

        # ── A2 lifecycle stage ────────────────────────────────────────
        "a2_static_init":         a2_static_init,
        "a2_constructor":         a2_constructor,
        "a2_test_method":         a2_test_method,
        "a2_unknown":             a2_unknown,
        "a2_other_stage":         a2_other_stage,

        # ── manual review ─────────────────────────────────────────────
        "a2_other_files":         a2_other_files,
    }

    result.update(a1_exc)   # a1_NullPointerException, a1_illegal_argument, a1_other, …
    result.update(a2_exc)   # a2_NullPointerException, a2_illegal_argument, a2_other, …
    return result


# =============================================================================
# LOADING
# =============================================================================

def load_all() -> dict:
    """
    Load each file declared in INPUT_FILES, extract values, return nested dict:
        results[exp_label][model_label] = extracted fields
    """
    results = defaultdict(dict)

    for (exp_label, model_label), path_str in INPUT_FILES.items():
        path = Path(path_str)
        if not path.exists():
            print(f"  ⚠  File not found, skipping: {path_str}")
            continue

        with open(path, encoding="utf-8") as fh:
            summary = json.load(fh)

        results[exp_label][model_label] = extract(summary)
        print(f"  ✓  {exp_label} / {model_label} ← {path.name}")

    return results


# =============================================================================
# EXPORT
# =============================================================================

def save_csv(results: dict, out_path: str):
    rows = []
    for exp, models in results.items():
        for model, fields in models.items():
            row = {"experiment": exp, "model": model}
            row.update({k: v for k, v in fields.items() if k != "a2_other_files"})
            rows.append(row)

    if not rows:
        print("  No data to write.")
        return

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=rows[0].keys())
        w.writeheader()
        w.writerows(rows)
    print(f"  CSV → {out_path}")


# =============================================================================
# MAIN
# =============================================================================

def main():
    print(f"\n  Extracting table values ...\n")
    results = load_all()

    if not results:
        return

    save_csv(results, OUTPUT_CSV)
    print(f"\n  Done.")

    # ── Quick console preview ─────────────────────────────────────────────
    exc_cols = list(_EXCEPTION_CATEGORIES.values()) + ["other"]

    print("\n  Preview:")
    for exp, models in sorted(results.items()):
        print(f"\n  {exp}")
        for model, fields in sorted(models.items()):
            print(f"    {model}")
            print(f"      Total files       : {fields['total_files']} "
                  f"(has asserts: {fields['a1_total_files']}, "
                  f"no asserts: {fields['a2_total_files']})")
            print(f"      Assert executed   : {fields['assert_executed']} "
                  f"(in try/catch: {fields['assert_executed_in_tc']})")
            print(f"      Failed before     : {fields['failed_before_assert']}")
            print(f"      A1 exception breakdown:")
            for col in exc_cols:
                v = fields.get(f"a1_{col}", 0)
                if v:
                    print(f"        {col}: {v}")
            print(f"      A2 exception breakdown:")
            for col in exc_cols:
                v = fields.get(f"a2_{col}", 0)
                if v:
                    print(f"        {col}: {v}")


if __name__ == "__main__":
    main()


  Extracting table values ...

  ✓  Exp3 (Minimal) / GPT-4o ← execution_rate_summary.json
  ✓  Exp3 (Minimal) / Qwen3-coder-480b ← execution_rate_summary.json
  ✓  Exp3 (Minimal) / GPT-OSS-120b ← execution_rate_summary.json
  ✓  Exp6 (Method) / GPT-4o ← execution_rate_summary.json
  ✓  Exp6 (Method) / Qwen3-coder-480b ← execution_rate_summary.json
  ✓  Exp6 (Method) / GPT-OSS-120b ← execution_rate_summary.json
  ✓  Exp7 (Class) / GPT-4o ← execution_rate_summary.json
  ✓  Exp7 (Class) / Qwen3-coder-480b ← execution_rate_summary.json
  ✓  Exp7 (Class) / GPT-OSS-120b ← execution_rate_summary.json
  CSV → /Volumes/Rachna-HD/AssertAnalysisResults/RQ3_table_values.csv

  Done.

  Preview:

  Exp3 (Minimal)
    GPT-4o
      Total files       : 168 (has asserts: 135, no asserts: 33)
      Assert executed   : 3 (in try/catch: 3)
      Failed before     : 132
      A1 exception breakdown:
        NoClassDefFoundError: 120
        NoSuchMethodError: 15
      A2 exception breakdown:
        NoCla